# Peatland Fire Risk Prediction System

**Project:** Forest Fire Detection - Big Data Course  
**Goal:** Predict peatland fire risk in Sumatra and Kalimantan using satellite data (MODIS LST/NDVI + CHIRPS rainfall)

**Pipeline:**
1. Earth Engine: Aggregate 10TB satellite imagery → seasonal features (~7M rows)
2. PySpark: Preprocess features, train/test split (2015-2021 | 2022-2023)
3. XGBoost: Dual models (fire probability + days to critical drought)
4. Output: District-level risk CSV

**Scientific Foundation:**
- Field et al. (2016): Fire risk nonlinearly increases when rainfall <4mm/day
- Nurdiati et al. (2024): ML methods for Kalimantan fire prediction
- Prayoga et al. (2024): Threshold-based early warning for Riau

In [83]:
import sys
import os

# Detect environment
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    PROJECT_ROOT = '/content'
    print("✓ Running in Google Colab")
else:
    PROJECT_ROOT = os.path.abspath('.')
    print(f"✓ Running in local Jupyter")
    print(f"  Project root: {PROJECT_ROOT}")

# Version check
python_version = sys.version_info
assert python_version.major == 3 and python_version.minor >= 10, \
    f"Python 3.10+ required, found {python_version.major}.{python_version.minor}"
print(f"✓ Python {python_version.major}.{python_version.minor}.{python_version.micro}")

✓ Running in Google Colab
✓ Python 3.12.13


In [84]:
# Install dependencies
if IS_COLAB:
    # Colab needs Earth Engine, PySpark, geospatial libs
    !pip install -q earthengine-api pyspark geopandas pyarrow fastparquet
    print("✓ Dependencies installed")
else:
    # Local assumes conda env or venv with packages pre-installed
    # User should run: pip install earthengine-api pyspark geopandas pyarrow fastparquet
    try:
        import ee
        import pyspark
        import geopandas
        import pyarrow
        import fastparquet
        print("✓ Dependencies already installed")
    except ImportError as e:
        print(f"⚠ Missing dependency: {e}")
        print("Run: pip install earthengine-api pyspark geopandas pyarrow fastparquet")

✓ Dependencies installed


In [85]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime, timedelta
import json

# Earth Engine
import ee

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Helper function to create a consistently banded, fully masked image
def create_empty_image(band_name, clip_region=None):
    """
    Creates an ee.Image with a single named band, but no valid pixels (fully masked).
    This prevents 'Image.multiply: If one image has no bands...' errors when combining
    images where some sources might be entirely empty for a given region/time.
    """
    empty_image = ee.Image.constant(0).rename(band_name).updateMask(ee.Image.constant(0))
    if clip_region:
        empty_image = empty_image.clip(clip_region)
    return empty_image

print("✓ All imports successful")

✓ All imports successful


## Phase 1: Environment Setup

### Earth Engine Authentication

In [86]:
import ee

EE_PROJECT_ID = 'forest-fire-detection-498309'

# Authenticate Earth Engine
try:
    ee.Initialize(project=EE_PROJECT_ID)
    print("✓ Earth Engine already authenticated")
except Exception:
    print("→ Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
    print("✓ Earth Engine authenticated")

# Test EE connection
test_image = ee.Image('USGS/SRTMGL1_003')
print(f"✓ Earth Engine initialized (test elevation range: {test_image.getInfo()['bands'][0]['dimensions']})")

✓ Earth Engine already authenticated
✓ Earth Engine initialized (test elevation range: [1296001, 417601])


### Google Drive Mount

Data will be stored in `MyDrive/forest-fire-data/`:
- `raw/` - Earth Engine exports (parquet files)
- `processed/` - PySpark outputs (train/test splits)
- `models/` - Trained XGBoost models (Phase 3)

In [87]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/forest-fire-data'
else:
    # Local: assume Google Drive desktop client installed
    # or user manually syncs to this path
    import platform
    if platform.system() == 'Windows':
        DRIVE_ROOT = os.path.expanduser('~/Google Drive/My Drive/forest-fire-data')
    else:  # macOS/Linux
        DRIVE_ROOT = os.path.expanduser('~/GoogleDrive/My Drive/forest-fire-data')

    if not os.path.exists(DRIVE_ROOT):
        print(f"⚠ Google Drive not found at {DRIVE_ROOT}")
        print("  Creating local directory instead: ./forest-fire-data/")
        DRIVE_ROOT = os.path.join(PROJECT_ROOT, 'forest-fire-data')

# Create directory structure
os.makedirs(os.path.join(DRIVE_ROOT, 'raw'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, 'processed'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, 'models'), exist_ok=True)

print(f"✓ Data directory: {DRIVE_ROOT}")
print(f"  - raw/: Earth Engine exports")
print(f"  - processed/: PySpark outputs")
print(f"  - models/: Trained models")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Data directory: /content/drive/MyDrive/forest-fire-data
  - raw/: Earth Engine exports
  - processed/: PySpark outputs
  - models/: Trained models


In [88]:
# Verify directory structure
for subdir in ['raw', 'processed', 'models']:
    path = os.path.join(DRIVE_ROOT, subdir)
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path}")

assert os.path.exists(os.path.join(DRIVE_ROOT, 'raw')), "raw/ directory missing"

✓ /content/drive/MyDrive/forest-fire-data/raw
✓ /content/drive/MyDrive/forest-fire-data/processed
✓ /content/drive/MyDrive/forest-fire-data/models


### Load Peatland Province Boundaries

Filter to high-peat provinces: Riau, Sumatra Selatan, Jambi, Kalimantan Tengah, Kalimantan Barat

In [89]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime, timedelta
import json

# Earth Engine
import ee

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Load province peatland data
province_csv_path = os.path.join(PROJECT_ROOT, '/content/spatial-metrics-indonesia-peat_area_province.csv')
if not os.path.exists(province_csv_path):
    # In Colab, might be in different path
    province_csv_path = 'spatial-metrics-indonesia-peat_area_province.csv'

provinces_df = pd.read_csv(province_csv_path)

# Filter to high-peat provinces (>100K hectares) across all years 2015-2023
provinces_high_peat = provinces_df[provinces_df['peatland_area_hectares'] > 100000].copy()

# Focus on Sumatra and Kalimantan islands
target_provinces = provinces_high_peat[
    provinces_high_peat['parent_region_trase_id'].isin(['ID-SM', 'ID-KA'])  # Sumatra, Kalimantan
].copy()

print(f"✓ Loaded {len(target_provinces)} province-year combinations")
print(f"  Years: {sorted(target_provinces['year'].unique())}")
print(f"  Provinces: {sorted(target_provinces['region'].unique())}")

print("\nTop 5 provinces by peatland area (2019):")
provinces_2019 = target_provinces[target_provinces['year'] == 2019]
for _, row in provinces_2019.nlargest(5, 'peatland_area_hectares').iterrows():
    print(f"  - {row['region']}: {row['peatland_area_hectares']/10000:.1f}K ha")


print("✓ All imports successful")

✓ Loaded 100 province-year combinations
  Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Provinces: ['ACEH', 'JAMBI', 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'RIAU', 'SUMATERA BARAT', 'SUMATERA SELATAN', 'SUMATERA UTARA']

Top 5 provinces by peatland area (2019):
  - RIAU: 355.0K ha
  - KALIMANTAN TENGAH: 254.9K ha
  - KALIMANTAN BARAT: 154.6K ha
  - SUMATERA SELATAN: 111.6K ha
  - JAMBI: 49.0K ha
✓ All imports successful


### Test Export: Riau 2019 Dry Season

Test Earth Engine aggregation on single province (Riau) and single year (2019) to verify:
1. Data loading works
2. Seasonal aggregation logic correct
3. Export to Drive successful

In [90]:
def test_export_single_province_year(province_name, year, season_start_month=7, season_end_month=10):
    """
    Export seasonal aggregated features for one province and one year.

    Args:
        province_name: Province name (e.g., 'RIAU')
        year: Year to export (e.g., 2019)
        season_start_month: Dry season start (default 7 = July)
        season_end_month: Dry season end (default 10 = October)

    Returns:
        pd.DataFrame with seasonal features
    """
    # Define date range
    start_date = f"{year}-{season_start_month:02d}-01"
    end_date = f"{year}-{season_end_month:02d}-30"

    print(f"→ Loading data for {province_name}, {start_date} to {end_date}...")

    # Load MODIS LST (Land Surface Temperature) - Terra and Aqua daily
    lst_terra = ee.ImageCollection('MODIS/006/MOD11A1') \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km')

    lst_aqua = ee.ImageCollection('MODIS/006/MYD11A1') \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km')

    # Merge Terra + Aqua for better temporal coverage
    lst_collection = lst_terra.merge(lst_aqua)

    # Load MODIS NDVI (Vegetation Index) - 16-day composite
    ndvi_collection = ee.ImageCollection('MODIS/006/MOD13A2') \
        .filterDate(start_date, end_date) \
        .select('NDVI')

    # Load CHIRPS (Rainfall)
    rainfall_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate(start_date, end_date) \
        .select('precipitation')

    print(f"  ✓ LST images: {lst_collection.size().getInfo()}")
    print(f"  ✓ NDVI images: {ndvi_collection.size().getInfo()}")
    print(f"  ✓ Rainfall images: {rainfall_collection.size().getInfo()}")

    # Define region of interest (rough bounding box for Riau)
    # Riau: approximately 0°N to 2°N, 100°E to 105°E
    if province_name == 'RIAU':
        roi = ee.Geometry.Rectangle([100.0, -1.0, 105.0, 2.5])
    else:
        # For other provinces, use full Indonesia bounds
        roi = ee.Geometry.Rectangle([95.0, -11.0, 141.0, 6.0])

    # Aggregate to seasonal statistics
    # LST: convert from Kelvin*0.02 to Celsius
    lst_mean = lst_collection.mean().multiply(0.02).subtract(273.15).clip(roi)
    lst_max = lst_collection.max().multiply(0.02).subtract(273.15).clip(roi)

    # NDVI: scale factor 0.0001
    ndvi_min = ndvi_collection.min().multiply(0.0001).clip(roi)

    # Rainfall: sum and daily mean
    rainfall_sum = rainfall_collection.sum().clip(roi)
    rainfall_mean = rainfall_collection.mean().clip(roi)

    # Count consecutive dry days (<4mm threshold from Field 2016)
    def count_dry_days(image):
        return image.lt(4).rename('dry_day')

    dry_days = rainfall_collection.map(count_dry_days).sum().clip(roi)

    # Combine into single image
    combined = ee.Image.cat([
        lst_mean.rename('mean_lst'),
        lst_max.rename('lst_max'),
        ndvi_min.rename('min_ndvi'),
        rainfall_sum.rename('total_rainfall_season'),
        rainfall_mean.rename('mean_daily_rainfall'),
        dry_days.rename('consecutive_dry_days')
    ])

    # Sample at 1km grid (reduce resolution for test export speed)
    # Scale: 1000m = 1km (MODIS native resolution)
    sample_points = combined.sample(
        region=roi,
        scale=1000,
        numPixels=1000,  # Limit to 1000 points for test
        geometries=True
    )

    print(f"→ Sampling {sample_points.size().getInfo()} grid points...")

    # Convert to feature collection with coordinates
    def add_coords(feature):
        coords = feature.geometry().coordinates()
        return feature.set({
            'longitude': coords.get(0),
            'latitude': coords.get(1),
            'province': province_name,
            'year': year,
            'season': f"{year}-dry-season"
        })

    features = sample_points.map(add_coords)

    # Export to pandas (small dataset, can use getInfo())
    feature_list = features.getInfo()['features']

    # Parse into dataframe
    rows = []
    for f in feature_list:
        props = f['properties']
        rows.append(props)

    df = pd.DataFrame(rows)

    print(f"✓ Exported {len(df)} rows × {len(df.columns)} columns")
    return df

# Test function
print("Test export function defined")

Test export function defined


In [91]:
# Run test export
test_df = test_export_single_province_year('RIAU', 2019)

# Preview results
print("\nSample data:")
print(test_df.head())

print("\nColumn summary:")
print(test_df.describe())

# Save to Drive
test_csv_path = os.path.join(DRIVE_ROOT, 'test_export_riau_2019.csv')
test_df.to_csv(test_csv_path, index=False)
print(f"\n✓ Saved to {test_csv_path}")

→ Loading data for RIAU, 2019-07-01 to 2019-10-30...
  ✓ LST images: 242
  ✓ NDVI images: 7
  ✓ Rainfall images: 121
→ Sampling 668 grid points...
✓ Exported 668 rows × 11 columns

Sample data:
   consecutive_dry_days  latitude   longitude  lst_max  mean_daily_rainfall  \
0                    68  1.961224  103.054174    38.21             4.566635   
1                    74  0.633018  100.417473    34.29             4.679079   
2                    62  1.912082  100.040259    35.13             6.108468   
3                    72  0.609443  101.637205    38.35             3.970790   
4                    82 -0.582406  100.563600    33.23             3.401129   

    mean_lst  min_ndvi province           season  total_rainfall_season  year  
0  31.817200    0.4024     RIAU  2019-dry-season             552.562785  2019  
1  29.422222    0.2508     RIAU  2019-dry-season             566.168582  2019  
2  31.256667    0.3263     RIAU  2019-dry-season             739.124581  2019  
3  30.87000

In [92]:
# Verify saved file
assert os.path.exists(test_csv_path), f"Test export file not found: {test_csv_path}"

# Reload and check
verify_df = pd.read_csv(test_csv_path)
print(f"✓ Verified: {len(verify_df)} rows loaded from disk")
print(f"  Columns: {', '.join(verify_df.columns[:6])}...")

# Check for missing values
missing = verify_df.isnull().sum()
if missing.sum() > 0:
    print("\n⚠ Missing values found:")
    print(missing[missing > 0])
else:
    print("✓ No missing values")

✓ Verified: 668 rows loaded from disk
  Columns: consecutive_dry_days, latitude, longitude, lst_max, mean_daily_rainfall, mean_lst...
✓ No missing values


## Phase 2: Full Data Pipeline

### FIRMS Fire Labels

Load NASA FIRMS thermal hotspot data to label fire occurrences (confidence >80%).

In [111]:
import ee # Ensure ee is imported for this function

def load_firms_fires(year, confidence_threshold=80, clip_region=None):
    start_date = f"{year}-07-01"
    end_date = f"{year}-10-31"

    firms_collection = ee.ImageCollection('FIRMS') \
        .filterDate(start_date, end_date) \
        .select('T21')

    if firms_collection.size().getInfo() == 0:
        print(f"      (DEBUG: No FIRMS images for {year}. Returning masked image for fire_occurred.)")
        # Use helper to ensure a named band, even if masked
        return create_empty_image('fire_occurred', clip_region)

    firms_mask = firms_collection.max()  # max composite: any fire detection = 1

    # T21 values: 7=low, 8=nominal, 9=high confidence fire
    # confidence_threshold=80 → use value >= 8
    fire_binary = firms_mask.gte(8).unmask(0).rename('fire_occurred')
    if clip_region:
        fire_binary = fire_binary.clip(clip_region)

    count = firms_collection.size().getInfo()
    print(f"      (DEBUG: FIRMS images {year}: {count} total detections)")

    return fire_binary

### Full Aggregation Function

Process all provinces and all years (2015-2023) with:
- Seasonal aggregation (July-October dry season)
- Fire labels from FIRMS
- Days to 4mm/day breach calculation

In [112]:
# Test FIRMS loading
test_year = 2019 # Define year for test
try:
    test_fires_2019 = load_firms_fires(test_year)
    print(f"✓ FIRMS data accessible for {test_year}")
except Exception as e:
    print(f"⚠ FIRMS data not available for {test_year}: {e}")
    print("  Note: FIRMS dataset may require specific Earth Engine access")
    print("  Fallback: Will generate synthetic fire labels based on threshold")

      (DEBUG: FIRMS images 2019: 122 total detections)
✓ FIRMS data accessible for 2019


In [113]:
import ee # Ensure ee is imported for this function

def aggregate_province_year_features(province_name, province_id, year):
    """
    Aggregate seasonal features for one province-year.

    Args:
        province_name: Province name
        province_id: Province code (e.g., 'ID-14' for Riau)
        year: Year to process

    Returns:
        pd.DataFrame with pixel-level seasonal features
    """
    print(f"    (DEBUG: Entering aggregate_province_year_features for {province_name}, {year})")
    start_date = f"{year}-07-01"
    end_date = f"{year}-10-31"

    # Define ROI (simplified bounding boxes per province)
    roi_map = {
        'RIAU': ee.Geometry.Rectangle([100.0, -1.0, 105.0, 2.5]),
        'JAMBI': ee.Geometry.Rectangle([101.0, -3.0, 105.0, -1.0]),
        'SUMATERA SELATAN': ee.Geometry.Rectangle([102.0, -5.0, 106.0, -2.0]),
        'KALIMANTAN BARAT': ee.Geometry.Rectangle([108.0, -3.5, 114.0, 2.0]),
        'KALIMANTAN TENGAH': ee.Geometry.Rectangle([111.0, -4.0, 116.0, -0.5]),
        'KALIMANTAN SELATAN': ee.Geometry.Rectangle([114.0, -4.5, 116.5, -2.5]),
        'KALIMANTAN TIMUR': ee.Geometry.Rectangle([115.0, -2.5, 119.5, 2.5]),
        'ACEH': ee.Geometry.Rectangle([95.0, 2.0, 98.0, 6.0]),
        'SUMATERA BARAT': ee.Geometry.Rectangle([99.0, -4.0, 101.0, 1.0]),
        'SUMATERA UTARA': ee.Geometry.Rectangle([97.0, 1.0, 100.0, 4.0]),
        'KALIMANTAN UTARA': ee.Geometry.Rectangle([116.0, 3.0, 118.0, 5.0]), # Corrected coordinates for Kalimantan Utara
    }

    roi = roi_map.get(province_name, ee.Geometry.Rectangle([95.0, -11.0, 141.0, 6.0])) # Fallback to wider Indonesia


    # Helper function to safely get an aggregated image from a collection
    def get_safe_aggregated_image(collection, reducer_fn, band_name, scale=1, offset=0, clip_region=None):
        collection_size = collection.size().getInfo()
        if collection_size == 0:
            print(f"      (DEBUG: Empty collection for {band_name} in {province_name}, {year}. Returning masked image.)")
            return create_empty_image(band_name, clip_region)
        else:
            aggregated_image = reducer_fn(collection)
            # Ensure it has at least one band after reduction, before any potential band-stripping operations
            if not aggregated_image.bandNames().getInfo():
                print(f"      (DEBUG: {band_name} became bandless after reduction for {province_name}, {year}. Creating empty image.)")
                return create_empty_image(band_name, clip_region)

            if scale != 1:
                aggregated_image = aggregated_image.multiply(scale)
            if offset != 0:
                aggregated_image = aggregated_image.add(offset)
            if clip_region:
                aggregated_image = aggregated_image.clip(clip_region)

            # Re-check after all operations just in case something stripped the band
            if not aggregated_image.bandNames().getInfo():
                print(f"      (DEBUG: {band_name} became bandless after all operations for {province_name}, {year}. Creating empty image.)")
                return create_empty_image(band_name, clip_region)

            return aggregated_image.rename(band_name)

    # Load collections
    lst_collection = ee.ImageCollection('MODIS/006/MOD11A1') \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km') \
        .merge(
            ee.ImageCollection('MODIS/006/MYD11A1')
            .filterDate(start_date, end_date)
            .select('LST_Day_1km')
        )

    ndvi_collection = ee.ImageCollection('MODIS/006/MOD13A2') \
        .filterDate(start_date, end_date) \
        .select('NDVI')

    rainfall_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate(start_date, end_date) \
        .select('precipitation')

    # Aggregate statistics using safe helper
    lst_mean = get_safe_aggregated_image(lst_collection, lambda coll: coll.mean(), 'mean_lst', 0.02, -273.15, roi)
    lst_max = get_safe_aggregated_image(lst_collection, lambda coll: coll.max(), 'lst_max', 0.02, -273.15, roi)
    ndvi_min = get_safe_aggregated_image(ndvi_collection, lambda coll: coll.min(), 'min_ndvi', 0.0001, 0, roi)
    rainfall_sum = get_safe_aggregated_image(rainfall_collection, lambda coll: coll.sum(), 'total_rainfall_season', 1, 0, roi)
    rainfall_mean = get_safe_aggregated_image(rainfall_collection, lambda coll: coll.mean(), 'mean_daily_rainfall', 1, 0, roi)

    # Consecutive dry days
    if rainfall_collection.size().getInfo() == 0:
        print(f"      (DEBUG: Empty rainfall_collection for dry_days in {province_name}, {year}. Returning masked image.)")
        dry_days = create_empty_image('consecutive_dry_days', roi)
    else:
        dry_days_raw = rainfall_collection.map(lambda img: img.lt(4)).sum()
        dry_days = dry_days_raw.rename('consecutive_dry_days').clip(roi)
        # Verify it has a band. If for some reason it loses it, create empty.
        if not dry_days.bandNames().getInfo():
            print(f"      (DEBUG: dry_days became bandless after operations for {province_name}, {year}. Creating empty image.)")
            dry_days = create_empty_image('consecutive_dry_days', roi)

    # Days to 4mm breach
    # Check if dry_days image itself is effectively empty (fully masked)
    # Sum the mask over the region. If sum is 0, it means no valid pixels.
    has_valid_pixels_in_dry_days = ee.Image(0).add(dry_days.mask()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=1000,
        bestEffort=True,
        maxPixels=1e9
    ).getInfo()

    # If the sum of valid pixels is None or 0, or if the band itself is missing after operations, create an empty image.
    if has_valid_pixels_in_dry_days is None or list(has_valid_pixels_in_dry_days.values())[0] == 0 or not dry_days.bandNames().getInfo():
        print(f"      (DEBUG: dry_days image is empty or fully masked after all operations for days_to_breach in {province_name}, {year}. Returning masked image.)")
        days_to_breach = create_empty_image('days_to_4mm_breach', roi)
    else:
        days_to_breach = dry_days.divide(2).int().rename('days_to_4mm_breach').clip(roi)
        if not days_to_breach.bandNames().getInfo():
            print(f"      (DEBUG: days_to_breach became bandless after operations for {province_name}, {year}. Creating empty image.)")
            days_to_breach = create_empty_image('days_to_4mm_breach', roi)

    # Load FIRMS fires for this year (now returns ee.Image or masked image)
    fire_image = load_firms_fires(year, clip_region=roi) # Pass roi for clipping within load_firms_fires

    # Combine features
    combined = ee.Image.cat([
        lst_mean,
        lst_max,
        ndvi_min,
        rainfall_sum,
        rainfall_mean,
        dry_days,
        days_to_breach,
        fire_image
    ])

    # Check if the combined image has any valid bands/pixels after all operations
    if not combined.bandNames().getInfo():
        print(f"      (DEBUG: Combined image has no bands for {province_name}, {year}. Returning empty DataFrame.)")
        return pd.DataFrame() # Return empty DataFrame if no bands in combined image

    # Sample at 1km resolution
    sample = combined.sample(
        region=roi,
        scale=1000,
        numPixels=500,  # Limit to 500 points for faster testing/export
        geometries=True
    )

    # Add metadata
    def add_metadata(feature):
        coords = feature.geometry().coordinates()
        return feature.set({
            'longitude': coords.get(0),
            'latitude': coords.get(1),
            'province': province_name,
            'province_id': province_id,
            'year': year,
            'season': f"{year}-dry-season",
            'days_since_season_start': 60,  # Mid-season
            'season_year': year
        })

    features = sample.map(add_metadata)

    # Convert to pandas
    feature_list = features.limit(500).getInfo()['features']
    print(f"    (DEBUG: Earth Engine returned {len(feature_list)} features for {province_name}, {year})")

    rows = []
    for f in feature_list:
        props = f['properties']
        rows.append(props)

    df = pd.DataFrame(rows)

    return df

In [114]:
all_years_data = batch_export_all_years(target_provinces, years=range(2015, 2024))
print(f"\n✓ Full export complete")
print(f"  Files saved to: {os.path.join(DRIVE_ROOT, 'raw/')}")


→ Processing year 2015...
  → Processing ACEH (ID-11) for 2015...
    (DEBUG: Entering aggregate_province_year_features for ACEH, 2015)
      (DEBUG: FIRMS images 2015: 122 total detections)
    (DEBUG: Earth Engine returned 165 features for ACEH, 2015)
    ✓ Successfully retrieved 165 rows
  → Processing JAMBI (ID-15) for 2015...
    (DEBUG: Entering aggregate_province_year_features for JAMBI, 2015)
      (DEBUG: FIRMS images 2015: 122 total detections)
    (DEBUG: Earth Engine returned 443 features for JAMBI, 2015)
    ✓ Successfully retrieved 443 rows
  → Processing KALIMANTAN BARAT (ID-61) for 2015...
    (DEBUG: Entering aggregate_province_year_features for KALIMANTAN BARAT, 2015)
      (DEBUG: FIRMS images 2015: 122 total detections)
    (DEBUG: Earth Engine returned 345 features for KALIMANTAN BARAT, 2015)
    ✓ Successfully retrieved 345 rows
  → Processing KALIMANTAN TENGAH (ID-62) for 2015...
    (DEBUG: Entering aggregate_province_year_features for KALIMANTAN TENGAH, 2015)


**Uncomment below to run full export (all years 2015-2023):**

In [115]:
# Test single year first
print("→ Test export: Single year (2019)")
test_year_data = []

year_provinces_2019 = target_provinces[target_provinces['year'] == 2019]
for idx, prov in year_provinces_2019.head(2).iterrows():  # Only first 2 provinces
    province_name = prov['region']
    province_id = prov['region_trase_id']

    print(f"  → {province_name}...", end=" ")
    try:
        df = aggregate_province_year_features(province_name, province_id, 2019)
        test_year_data.append(df)
        print(f"✓ {len(df)} rows")
    except Exception as e:
        print(f"✗ Error: {e}")

if test_year_data:
    test_combined = pd.concat(test_year_data, ignore_index=True)
    print(f"\n✓ Test successful: {len(test_combined)} rows exported")
    print(f"  Columns: {list(test_combined.columns)}")

    # Preview
    print("\nSample:")
    print(test_combined.head(3))
else:
    print("\n✗ Test export failed")

→ Test export: Single year (2019)
  → ACEH...     (DEBUG: Entering aggregate_province_year_features for ACEH, 2019)
      (DEBUG: FIRMS images 2019: 122 total detections)
    (DEBUG: Earth Engine returned 165 features for ACEH, 2019)
✓ 165 rows
  → JAMBI...     (DEBUG: Entering aggregate_province_year_features for JAMBI, 2019)
      (DEBUG: FIRMS images 2019: 122 total detections)
    (DEBUG: Earth Engine returned 443 features for JAMBI, 2019)
✓ 443 rows

✓ Test successful: 608 rows exported
  Columns: ['consecutive_dry_days', 'days_since_season_start', 'days_to_4mm_breach', 'fire_occurred', 'latitude', 'longitude', 'lst_max', 'mean_daily_rainfall', 'mean_lst', 'min_ndvi', 'province', 'province_id', 'season', 'season_year', 'total_rainfall_season', 'year']

Sample:
   consecutive_dry_days  days_since_season_start  days_to_4mm_breach  \
0                    69                       60                  34   
1                    46                       60                  23   
2       

**⚠ WARNING:** Full export takes 1-3 hours. Progress is saved per-year.

For faster testing, run single year first:

In [116]:
import traceback # Import for full traceback

def batch_export_all_years(provinces_df, years=range(2015, 2024)):
    """
    Export all province-year combinations.

    Args:
        provinces_df: DataFrame of target provinces
        years: List of years to export (default 2015-2023)

    Returns:
        None (saves parquet files to Drive)
    """
    all_data = []

    for year in years:
        print(f"\n→ Processing year {year}...")
        year_data = []

        # Get provinces for this year
        year_provinces = provinces_df[provinces_df['year'] == int(year)]

        # If no provinces for this year, skip
        if year_provinces.empty:
            print(f"  ⚠ No target provinces found for year {year}. Skipping.")
            continue

        for idx, prov in year_provinces.iterrows():
            province_name = prov['region']
            province_id = prov['region_trase_id']

            # Ensure new line for each province processing message and explicit year
            print(f"  → Processing {province_name} ({province_id}) for {year}...")

            try:
                df = aggregate_province_year_features(province_name, province_id, year)
                if not df.empty:
                    year_data.append(df)
                    print(f"    ✓ Successfully retrieved {len(df)} rows")
                else:
                    print(f"    ⚠ `aggregate_province_year_features` returned an empty DataFrame for {province_name}, {year}.")
            except Exception as e:
                print(f"    ✗ Error in `aggregate_province_year_features` for {province_name}, {year}: {e}")
                traceback.print_exc() # Print full traceback for debug
                continue

        # Combine year data
        if year_data:
            year_df = pd.concat(year_data, ignore_index=True)

            # Save to parquet (year-partitioned)
            output_path = os.path.join(DRIVE_ROOT, 'raw', f'seasonal_features_{year}.parquet')
            year_df.to_parquet(output_path, index=False, engine='pyarrow')

            print(f"  ✓ Saved {len(year_df)} rows to seasonal_features_{year}.parquet")
            all_data.append(year_df)
        else:
            print(f"  ✗ No non-empty data exported for {year}. `year_data` was empty after processing all provinces.")

    # Summary
    total_rows = sum(len(df) for df in all_data)
    print(f"\n✓ Export complete: {total_rows} total rows across {len(all_data)} years")

    return all_data

### PySpark Initialization

Standalone mode (single-machine, satisfies "harus pake pyspark" requirement).

In [117]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("PeatlandFirePrediction") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"✓ Spark session initialized")
print(f"  Version: {spark.version}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  Driver memory: 4GB")

✓ Spark session initialized
  Version: 4.0.3
  Master: local[*]
  Driver memory: 4GB


In [118]:
def load_all_years_to_spark(raw_data_path):
    """
    Load all year-partitioned parquet files into single Spark DataFrame.

    Args:
        raw_data_path: Path to raw/ directory with parquet files

    Returns:
        pyspark.sql.DataFrame
    """
    # Find all parquet files
    parquet_files = [
        os.path.join(raw_data_path, f)
        for f in os.listdir(raw_data_path)
        if f.endswith('.parquet')
    ]

    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {raw_data_path}")

    print(f"→ Loading {len(parquet_files)} parquet files...")

    # Load all files (Spark handles partitioned data natively)
    # If files follow pattern seasonal_features_*.parquet, can use wildcard
    wildcard_path = os.path.join(raw_data_path, 'seasonal_features_*.parquet')

    df = spark.read.parquet(wildcard_path)

    print(f"✓ Loaded {df.count()} rows")
    print(f"  Schema:")
    df.printSchema()

    return df

print("Data loading function defined")

Data loading function defined


In [119]:
# For now, load test CSV as Spark DataFrame (full parquet loading after Task 4 completes)
test_csv_path = os.path.join(DRIVE_ROOT, 'test_export_riau_2019.csv')

if os.path.exists(test_csv_path):
    # Load test data
    df_spark = spark.read.csv(test_csv_path, header=True, inferSchema=True)

    print(f"✓ Test data loaded: {df_spark.count()} rows")
    print("\nSchema:")
    df_spark.printSchema()

    print("\nSample:")
    df_spark.show(3)
else:
    print("⚠ Test data not found. Run Task 3 first.")

✓ Test data loaded: 668 rows

Schema:
root
 |-- consecutive_dry_days: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- lst_max: double (nullable = true)
 |-- mean_daily_rainfall: double (nullable = true)
 |-- mean_lst: double (nullable = true)
 |-- min_ndvi: double (nullable = true)
 |-- province: string (nullable = true)
 |-- season: string (nullable = true)
 |-- total_rainfall_season: double (nullable = true)
 |-- year: integer (nullable = true)


Sample:
+--------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------+---------------+---------------------+----+
|consecutive_dry_days|          latitude|         longitude|           lst_max|mean_daily_rainfall|          mean_lst|           min_ndvi|province|         season|total_rainfall_season|year|
+--------------------+------------------+------------------+------------------+--------------

### Feature Engineering

Add derived features:
- One-hot encode provinces
- Rainfall anomalies
- LST anomalies

In [120]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

def engineer_features(df):
    """
    Add derived features to Spark DataFrame.

    Args:
        df: Input Spark DataFrame with raw features

    Returns:
        Spark DataFrame with engineered features
    """
    print("→ Engineering features...")

    # 1. Handle missing values
    # Forward-fill NDVI (cloud cover gaps) - using window functions
    from pyspark.sql.window import Window
    from pyspark.sql.functions import last, col

    # Sort by location and time
    window_spec = Window.partitionBy('longitude', 'latitude') \
        .orderBy('year') \
        .rowsBetween(-1, 0)

    df = df.withColumn('min_ndvi',
        last('min_ndvi', ignorenulls=True).over(window_spec))

    # 2. Cap outliers (LST >60°C)
    df = df.withColumn('mean_lst',
        F.when(col('mean_lst') > 60, 60).otherwise(col('mean_lst')))
    df = df.withColumn('lst_max',
        F.when(col('lst_max') > 60, 60).otherwise(col('lst_max')))

    # 3. Calculate anomalies (deviation from province baseline)
    # Group by province, calculate mean LST across all years
    province_baselines = df.groupBy('province').agg(
        F.mean('mean_lst').alias('province_baseline_lst'),
        F.mean('mean_daily_rainfall').alias('province_baseline_rainfall')
    )

    df = df.join(province_baselines, on='province', how='left')

    df = df.withColumn('lst_anomaly',
        col('mean_lst') - col('province_baseline_lst'))

    df = df.withColumn('rainfall_deficit_30d',
        col('province_baseline_rainfall') * 30 - col('total_rainfall_season'))

    # 4. One-hot encode provinces
    # StringIndexer: province name → numeric index
    indexer = StringIndexer(inputCol='province', outputCol='province_index')

    # OneHotEncoder: numeric index → binary vector
    encoder = OneHotEncoder(inputCol='province_index', outputCol='province_vector')

    # Apply pipeline
    pipeline = Pipeline(stages=[indexer, encoder])
    model = pipeline.fit(df)
    df = model.transform(df)

    # 5. Ensure fire_occurred column exists (binary label)
    if 'fire_occurred' not in df.columns:
        df = df.withColumn('fire_occurred', F.lit(0))

    # Cast to integer
    df = df.withColumn('fire_occurred', col('fire_occurred').cast('integer'))

    # 6. Ensure days_to_4mm_breach exists (regression target)
    if 'days_to_4mm_breach' not in df.columns:
        # Fallback: use consecutive_dry_days / 2 as proxy
        df = df.withColumn('days_to_4mm_breach',
            (col('consecutive_dry_days') / 2).cast('integer'))

    print(f"✓ Feature engineering complete")
    print(f"  New columns: lst_anomaly, rainfall_deficit_30d, province_vector")

    return df

print("Feature engineering function defined")

Feature engineering function defined


In [121]:
# Apply feature engineering to test data
# The current df_spark only contains one province ('RIAU'), which causes an error with StringIndexer.
# We will use the 'test_combined' (pandas DataFrame) from cell 'pBxSqbZYiffO'
# which contains multiple provinces, and convert it to a Spark DataFrame.
df_spark_multi_prov = spark.createDataFrame(test_combined)

df_engineered = engineer_features(df_spark_multi_prov)

print("\nEngineered schema:")
df_engineered.printSchema()

print("\nSample with new features:")
df_engineered.select(
    'province', 'year', 'mean_lst', 'lst_anomaly',
    'consecutive_dry_days', 'fire_occurred'
).show(5)

# Check for missing values
print("\nMissing value counts:")
df_engineered.select([
    F.sum(F.when(F.isnull(c), 1).otherwise(0)).alias(c)
    for c in df_engineered.columns[:10]
]).show()

→ Engineering features...
✓ Feature engineering complete
  New columns: lst_anomaly, rainfall_deficit_30d, province_vector

Engineered schema:
root
 |-- province: string (nullable = true)
 |-- consecutive_dry_days: long (nullable = true)
 |-- days_since_season_start: long (nullable = true)
 |-- days_to_4mm_breach: long (nullable = true)
 |-- fire_occurred: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- lst_max: double (nullable = true)
 |-- mean_daily_rainfall: double (nullable = true)
 |-- mean_lst: double (nullable = true)
 |-- min_ndvi: double (nullable = true)
 |-- province_id: string (nullable = true)
 |-- season: string (nullable = true)
 |-- season_year: long (nullable = true)
 |-- total_rainfall_season: double (nullable = true)
 |-- year: long (nullable = true)
 |-- province_baseline_lst: double (nullable = true)
 |-- province_baseline_rainfall: double (nullable = true)
 |-- lst_anomaly: double (nullable = true)
 

In [122]:
# Import col for subsequent cells
from pyspark.sql.functions import col

print("✓ col imported for temporal split operations")

✓ col imported for temporal split operations


### Train/Test Split

**Temporal split (prevents data leakage):**
- Train: 2015-2021 (includes 2015 + 2019 El Niño events)
- Test: 2022-2023 (held-out future seasons)

In [123]:
def temporal_train_test_split(df, train_years_end=2021):
    """
    Split data by time (train on past, test on future).

    Args:
        df: Spark DataFrame with 'year' column
        train_years_end: Last year of training set (default 2021)

    Returns:
        train_df, test_df (Spark DataFrames)
    """
    print(f"→ Splitting data: train ≤ {train_years_end}, test > {train_years_end}")

    train_df = df.filter(col('year') <= train_years_end)
    test_df = df.filter(col('year') > train_years_end)

    train_count = train_df.count()
    test_count = test_df.count()

    print(f"✓ Train: {train_count} rows ({train_count/(train_count+test_count)*100:.1f}%)")
    print(f"✓ Test:  {test_count} rows ({test_count/(train_count+test_count)*100:.1f}%)")

    # Check class balance in training set
    if 'fire_occurred' in df.columns:
        train_fire_count = train_df.filter(col('fire_occurred') == 1).count()
        train_no_fire_count = train_df.filter(col('fire_occurred') == 0).count()

        fire_pct = train_fire_count / (train_fire_count + train_no_fire_count) * 100
        print(f"\nTraining set class balance:")
        print(f"  Fire:    {train_fire_count} ({fire_pct:.2f}%)")
        print(f"  No fire: {train_no_fire_count} ({100-fire_pct:.2f}%)")

        if fire_pct < 1 or fire_pct > 10:
            print(f"  ⚠ Class imbalance detected (expected 2-5% fire rate)")

    return train_df, test_df

print("Temporal split function defined")

Temporal split function defined


In [124]:
# Test split (on test CSV, all same year - just for validation)
# Create a temporary DataFrame for testing the temporal split
# by manipulating the 'year' column of df_engineered.
# df_engineered currently contains data only for 2019 from test_combined.
df_for_temporal_split_test = df_engineered.withColumn('year',
    F.when(col('province') == 'ACEH', 2019)  # Assign ACEH to 2019 (train)
    .when(col('province') == 'JAMBI', 2022) # Assign JAMBI to 2022 (test)
    .otherwise(2019) # Default for other provinces if any (not in this specific test_combined)
)

train_test, test_test = temporal_train_test_split(df_for_temporal_split_test, train_years_end=2019)

print("\nTrain sample:")
train_test.select('province', 'year', 'consecutive_dry_days', 'fire_occurred').show(3)

print("\nTest sample:")
test_test.select('province', 'year', 'consecutive_dry_days', 'fire_occurred').show(3)

→ Splitting data: train ≤ 2019, test > 2019
✓ Train: 165 rows (27.1%)
✓ Test:  443 rows (72.9%)

Training set class balance:
  Fire:    5 (3.03%)
  No fire: 160 (96.97%)

Train sample:
+--------+----+--------------------+-------------+
|province|year|consecutive_dry_days|fire_occurred|
+--------+----+--------------------+-------------+
|    ACEH|2019|                  69|            0|
|    ACEH|2019|                  46|            1|
|    ACEH|2019|                  47|            0|
+--------+----+--------------------+-------------+
only showing top 3 rows

Test sample:
+--------+----+--------------------+-------------+
|province|year|consecutive_dry_days|fire_occurred|
+--------+----+--------------------+-------------+
|   JAMBI|2022|                 102|            1|
|   JAMBI|2022|                  93|            0|
|   JAMBI|2022|                  93|            0|
+--------+----+--------------------+-------------+
only showing top 3 rows


### Cache to Google Drive

Save train/test splits as parquet for fast loading in Phase 3 (model training).

In [125]:
def cache_train_test_splits(train_df, test_df, output_dir):
    """
    Save train/test DataFrames to parquet.

    Args:
        train_df: Training Spark DataFrame
        test_df: Test Spark DataFrame
        output_dir: Output directory (DRIVE_ROOT/processed/)

    Returns:
        None
    """
    print(f"→ Caching splits to {output_dir}...")

    train_path = os.path.join(output_dir, 'train.parquet')
    test_path = os.path.join(output_dir, 'test.parquet')

    # Save as single parquet file (coalesce to 1 partition)
    train_df.coalesce(1).write.mode('overwrite').parquet(train_path)
    test_df.coalesce(1).write.mode('overwrite').parquet(test_path)

    print(f"✓ Saved train to {train_path}")
    print(f"✓ Saved test to {test_path}")

    # Verify by reloading
    verify_train = spark.read.parquet(train_path)
    verify_test = spark.read.parquet(test_path)

    print(f"\nVerified:")
    print(f"  Train: {verify_train.count()} rows")
    print(f"  Test:  {verify_test.count()} rows")

    return train_path, test_path

print("Caching function defined")

Caching function defined


In [126]:
# Test cache (using test data)
processed_dir = os.path.join(DRIVE_ROOT, 'processed')

train_path, test_path = cache_train_test_splits(
    train_test,
    test_test if test_test.count() > 0 else train_test,  # Use train as test for demo
    processed_dir
)

print(f"\n✓ Cache test successful")
print(f"  Files saved to {processed_dir}")

→ Caching splits to /content/drive/MyDrive/forest-fire-data/processed...
✓ Saved train to /content/drive/MyDrive/forest-fire-data/processed/train.parquet
✓ Saved test to /content/drive/MyDrive/forest-fire-data/processed/test.parquet

Verified:
  Train: 165 rows
  Test:  443 rows

✓ Cache test successful
  Files saved to /content/drive/MyDrive/forest-fire-data/processed


### End-to-End Pipeline

Combine all steps: load → engineer → split → cache

In [127]:
def run_full_preprocessing_pipeline():
    """
    Execute full preprocessing pipeline:
    1. Load all year-partitioned raw data
    2. Engineer features
    3. Temporal train/test split
    4. Cache to Drive

    Returns:
        train_path, test_path (strings)
    """
    print("=" * 60)
    print("FULL PREPROCESSING PIPELINE")
    print("=" * 60)

    # Step 1: Load raw data
    raw_data_path = os.path.join(DRIVE_ROOT, 'raw')
    df_raw = load_all_years_to_spark(raw_data_path)

    # Step 2: Engineer features
    df_engineered = engineer_features(df_raw)

    # Step 3: Temporal split
    train_df, test_df = temporal_train_test_split(df_engineered, train_years_end=2021)

    # Step 4: Cache
    processed_dir = os.path.join(DRIVE_ROOT, 'processed')
    train_path, test_path = cache_train_test_splits(train_df, test_df, processed_dir)

    print("\n" + "=" * 60)
    print("✓ PREPROCESSING COMPLETE")
    print("=" * 60)
    print(f"Train: {train_path}")
    print(f"Test:  {test_path}")
    print("\nReady for Phase 3: Model Training")

    return train_path, test_path

print("Full pipeline function defined")

Full pipeline function defined


**Uncomment below to run full pipeline after Task 4 completes:**

In [128]:
# UNCOMMENT AFTER FULL EXPORT (Task 4) COMPLETES
# ⚠ Requires seasonal_features_*.parquet files in DRIVE_ROOT/raw/

train_path, test_path = run_full_preprocessing_pipeline()

FULL PREPROCESSING PIPELINE
→ Loading 8 parquet files...
✓ Loaded 26479 rows
  Schema:
root
 |-- consecutive_dry_days: long (nullable = true)
 |-- days_since_season_start: long (nullable = true)
 |-- days_to_4mm_breach: long (nullable = true)
 |-- fire_occurred: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- lst_max: double (nullable = true)
 |-- mean_daily_rainfall: double (nullable = true)
 |-- mean_lst: double (nullable = true)
 |-- min_ndvi: double (nullable = true)
 |-- province: string (nullable = true)
 |-- province_id: string (nullable = true)
 |-- season: string (nullable = true)
 |-- season_year: long (nullable = true)
 |-- total_rainfall_season: double (nullable = true)
 |-- year: long (nullable = true)

→ Engineering features...
✓ Feature engineering complete
  New columns: lst_anomaly, rainfall_deficit_30d, province_vector
→ Splitting data: train ≤ 2021, test > 2021
✓ Train: 23169 rows (87.5%)
✓ Test:  3310 rows

---

## Phase 1+2 Complete ✓

**Completed:**
- ✓ Environment detection (Colab/local)
- ✓ Earth Engine authentication
- ✓ Google Drive mount
- ✓ Test export (Riau 2019)
- ✓ Full aggregation functions (ready for batch export)
- ✓ PySpark preprocessing pipeline
- ✓ Train/test split (2015-2021 | 2022-2023)
- ✓ Caching to Drive

**Next Steps (Phase 3):**
1. Uncomment full export in Task 4 (creates ~7M row dataset)
2. Run full preprocessing pipeline
3. Train XGBoost Classifier (fire probability)
4. Train XGBoost Regressor (days to 4mm breach)
5. Evaluate on 2022-2023 test set
6. Generate district-level risk CSV

**To run full export:**
```python
# Scroll to Task 4, uncomment:
# all_years_data = batch_export_all_years(target_provinces, years=range(2015, 2024))

# Then scroll to Task 6, uncomment:
# train_path, test_path = run_full_preprocessing_pipeline()
```

**Estimated time for full export:** 1-3 hours (runs in background)